In [ ]:
!pip install nltk
!pip install Wordcloud
!pip install scikit-learn==1.5.2 
!pip install spacy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import spacy
import random
import gzip

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
# Download NLTK resources (once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

print("Setup complete.")

In [ ]:
ls

### Create your text below (at leaste few sentences in one string) and perform the following tasks:
tokenize annd lower case \
removes stop words and punctuations \
lemmatize \
stemming \
word cloud \
and sentiment analysis using VADER


### Load the dataset below from Assignment3 folder and perform the below tasks 

In [ ]:
import os
os.getcwd()

In [ ]:
os.chdir('/home/jovyan/Assignment3/')

In [ ]:
ls

In [ ]:
df = pd.read_csv('climate_policy_200.gz', compression='gzip')

In [ ]:
df.head()

In [ ]:
# TASK 2:
# 1. Print tokens for each statement.
# 2. Add/remove stopwords and observe changes.
# 3. Try stemming instead of lemmatization.

In [ ]:
# Preprocessing Function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    lemmas = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(lemmas)

In [ ]:
df["Clean"] = df["Statement"].apply(preprocess)

In [ ]:
# Print tokens for each statement
for i, text in enumerate(df["Statement"][:5]): # just for the first 5
    tokens = word_tokenize(text.lower())
    print(f"Statement {i+1} tokens:", tokens)

In [ ]:
# Modify stopwords
custom_stopwords = stop_words.copy()
custom_stopwords.add("however")     # add a stopword
custom_stopwords.discard("energy")  # remove a stopword if present

def preprocess_custom(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in custom_stopwords]
    lemmas = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(lemmas)

In [ ]:
df["Clean_Custom"] = df["Statement"].apply(preprocess_custom)
df[["Statement", "Clean", "Clean_Custom"]].head()

In [ ]:
# Stemming instead of lemmatization
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def preprocess_stem(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    stems = [stemmer.stem(t) for t in tokens]
    return " ".join(stems)



In [ ]:
df["Clean_Stemmed"] = df["Statement"].apply(preprocess_stem)
df[["Statement", "Clean", "Clean_Stemmed"]]

### WORD CLOUD

In [ ]:

text_corpus = " ".join(df["Clean"])
wc = WordCloud(width=800, height=400, background_color="white").generate(text_corpus)

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Climate/Energy Discourse")
plt.show()

# TASK:
# - Change the colour map.
# - Generate separate word clouds for each speaker.


In [ ]:
wc = WordCloud(width=800, height=400, background_color="white", colormap="plasma").generate(text_corpus)
### try diff color maps: cool, inferno, viridis, magma

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud with Plasma Colour Map")
plt.show()

In [ ]:
### for different speakers
for speaker in df["Speaker"].unique():
    speaker_text = " ".join(df[df["Speaker"] == speaker]["Clean"])
    
    wc = WordCloud(width=800, height=400, background_color="white", colormap="viridis").generate(speaker_text)
    
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Word Cloud for {speaker}")
    plt.show()

### TF-IDF

In [ ]:
corpus = df["Clean"].tolist()
vectorizer = TfidfVectorizer(max_features=10)
X = vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df

# task:
# - Increase max_features to 20.
# - Add a new statement and compare TF-IDF again.


In [ ]:
vectorizer = TfidfVectorizer(max_features=20)
X = vectorizer.fit_transform(df["Clean"])
pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()).head()

### TOPIC MODELLING (NMF)

In [ ]:
nmf = NMF(n_components=2, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

for idx, topic in enumerate(H):
    print(f"Topic {idx+1}: ",
          [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-5:]])

# TASK 5:
# - Change n_components to 3.
# - Interpret the new topics.


In [ ]:
### changing to 3 momponents
nmf = NMF(n_components=3, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

for idx, topic in enumerate(H):
    print(f"Topic {idx+1}:",
          [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-7:]])

### SENTIMENT ANALYSIS

In [ ]:
### VADER Sentiment

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

df["Sentiment"] = df["Statement"].apply(lambda x: sia.polarity_scores(x)["compound"])
df[["Speaker", "Statement", "Sentiment"]]

plt.figure(figsize=(8,5))
plt.bar(df["Speaker"], df["Sentiment"], color="teal")
plt.title("Sentiment of Statements")
plt.ylabel("Compound Sentiment Score")
plt.show()

#  TASK:
# - Add a sarcastic sentence and observe misclassification.
# - Compare sentiment before/after removing negations.

In [ ]:
##solution
df["Sentiment"] = df["Statement"].apply(lambda x: sia.polarity_scores(x)["compound"])
df["Sentiment"].describe()

In [ ]:
# Add a sarcastic sentence
sarcastic = "Oh great, another increase in energy prices. Fantastic job!"
sia.polarity_scores(sarcastic)

In [ ]:
# Example sentence with negation
text = "The government is not doing enough to support clean energy."
original = sia.polarity_scores(text)["compound"]
print("Original:", original)

In [ ]:
# Remove negation manually
no_negation = text.replace("not ", "")
modified = sia.polarity_scores(no_negation)["compound"]
print("Without negation:", modified)

In [ ]:
#another one
text = "This policy is good."
neg_text = "This policy is not good."

print("text:     ",sia.polarity_scores(text)["compound"])
print("neg_text: ",sia.polarity_scores(neg_text)["compound"])


### NAMED ENTITY RECOGNITION (NER)


In [ ]:
# Make sure the model is installed:
# Run in terminal: python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")

doc = nlp("In 2023, the UK government invested £2 billion in renewable energy projects across London and Scotland.")
[(ent.text, ent.label_) for ent in doc.ents]

#  TASK 7:
# - Run NER on each statement in the dataset.
# - Count how many entities of each type appear.

In [ ]:
entities = []
for text in df["Statement"]:
    doc = nlp(text)
    entities.extend([(ent.text, ent.label_) for ent in doc.ents])

entities[:10]

In [ ]:
# Convert to DataFrame for easy counting
ent_df = pd.DataFrame(entities, columns=["Entity", "Type"])

# Count entities by type
ent_counts = ent_df["Type"].value_counts()
ent_counts

In [ ]:
#show on charts
ent_counts.plot(kind="bar", figsize=(8,5), color="teal")
plt.title("Named Entity Counts by Type")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Final task
# ============================================================

# Using the dataset:
# 1. Identify the top 10 TF-IDF words.
# 2. Extract 2–3 topics.
# 3. Plot sentiment distribution.
# 4. Extract named entities.
# 5. Write a 3–4 sentence interpretation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df["Clean"].tolist()
vectorizer = TfidfVectorizer(max_features=10)
X = vectorizer.fit_transform(corpus)

top_words = vectorizer.get_feature_names_out()
top_words


In [ ]:
## extract 2 to 3 topics
from sklearn.decomposition import NMF

nmf = NMF(n_components=3, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

print("=== NMF Topics ===")
for idx, topic in enumerate(H):
    top_terms = [top_words[i] for i in topic.argsort()[-5:]]
    print(f"Topic {idx+1}: {top_terms}")




In [ ]:
###Plot sentiment distribution
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

df["Sentiment"] = df["Statement"].apply(lambda x: sia.polarity_scores(x)["compound"])

plt.figure(figsize=(8,5))
plt.hist(df["Sentiment"], bins=10, color="teal", edgecolor="black")
plt.title("Sentiment Distribution of Statements")
plt.xlabel("Compound Sentiment Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
##extract named entities
import spacy
nlp = spacy.load("en_core_web_sm")

entities = []
for text in df["Statement"]:
    doc = nlp(text)
    entities.extend([(ent.text, ent.label_) for ent in doc.ents])

ent_df = pd.DataFrame(entities, columns=["Entity", "Type"])
ent_counts = ent_df["Type"].value_counts()
ent_counts


In [ ]:
## visuals
ent_counts.plot(kind="bar", figsize=(8,5), color="purple")
plt.title("Named Entity Counts by Type")
plt.ylabel("Frequency")
plt.show()


In [ ]:
print("Lab complete!")